# P10.6-AI — Notebook 55: entrenamiento de estenosis central

Entrena el primer clasificador RSNA 2.5D sobre Sagittal T2/STIR. Usa train y validation del Notebook 54; el internal test queda reservado para el Notebook 56.

`humanReviewRequired=true` · `notClinicalDiagnosis=true`


In [ ]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util, subprocess, sys
packages = {"pydicom":"pydicom","timm":"timm","tqdm":"tqdm","sklearn":"scikit-learn"}
missing = [pkg for mod,pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])


In [ ]:
# 2) GPU y Drive
import getpass, json, os, subprocess
from pathlib import Path
import torch
from google.colab import drive  # type: ignore

if not torch.cuda.is_available():
    raise RuntimeError("Seleccioná GPU T4 o superior en el runtime de Colab.")
print({"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__})
drive.mount("/content/drive", force_remount=False)


In [ ]:
# 3) Clonar/actualizar la rama e importar el pipeline
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_REF = "enzo/p10-6-ai-rsna-findings"

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_ROOT)])
else:
    subprocess.check_call(["git","fetch","origin"], cwd=REPO_ROOT)
    subprocess.check_call(["git","checkout",REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git","pull","--ff-only"], cwd=REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_central_training import (
    TrainConfig, attach_coordinates, build_cache, ensure_local_subset,
    load_manifests, sha256_file, train,
)
print({"repoSha": subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_ROOT,text=True).strip()})


In [ ]:
# 4) Rutas y configuración
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
SPLIT_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook54_split"
MODEL_ROOT = PFI_ROOT / "models" / "P10_6_rsna_findings" / "central_stenosis_sagittal_t2_2p5d"
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"
RUN_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings" / "notebook55_training"
LOCAL_ROOT = Path("/content/RSNA_LUMBAR_DISC")
CACHE_ROOT = Path("/content/rsna_central_stenosis_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"
CFG = TrainConfig()

for path in (MODEL_ROOT, CHECKPOINT_ROOT, RUN_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print(CFG)


In [ ]:
# 5) Cargar manifests aprobados sin leer internal_test
train_manifest, validation_manifest, split_summary = load_manifests(SPLIT_ROOT)
print({
    "trainRows": len(train_manifest),
    "validationRows": len(validation_manifest),
    "trainStudies": train_manifest.study_id.nunique(),
    "validationStudies": validation_manifest.study_id.nunique(),
    "internalTestAccessed": False,
})


In [ ]:
# 6) Preparar localmente solo las series requeridas
token = ""
if not (LOCAL_ROOT / "train_label_coordinates.csv").is_file():
    token = getpass.getpass("Pegá tu KAGGLE_API_TOKEN (no se mostrará): " ).strip()
    if not token:
        raise RuntimeError("No se ingresó token de Kaggle.")

ensure_local_subset(train_manifest, validation_manifest, LOCAL_ROOT, COMPETITION, token)
token = ""
print({"localRoot": str(LOCAL_ROOT), "ready": True})


In [ ]:
# 7) Unir coordenadas RSNA y construir cache 2.5D
train_samples = attach_coordinates(train_manifest, LOCAL_ROOT)
validation_samples = attach_coordinates(validation_manifest, LOCAL_ROOT)

build_cache(train_samples, CACHE_ROOT, "train", CFG)
build_cache(validation_samples, CACHE_ROOT, "validation", CFG)

print({
    "trainSamples": len(train_samples),
    "validationSamples": len(validation_samples),
    "trainClasses": train_samples.severity.value_counts().to_dict(),
    "validationClasses": validation_samples.severity.value_counts().to_dict(),
})


In [ ]:
# 8) Entrenar y guardar checkpoints
manifest_hashes = {
    "train_manifest.csv": sha256_file(SPLIT_ROOT / "train_manifest.csv"),
    "validation_manifest.csv": sha256_file(SPLIT_ROOT / "validation_manifest.csv"),
    "split_summary.json": sha256_file(SPLIT_ROOT / "split_summary.json"),
}
result = train(
    train_samples,
    validation_samples,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    RUN_ROOT,
    manifest_hashes,
    CFG,
)
print(json.dumps(result, indent=2))


In [ ]:
# 9) Gate para Notebook 56
best_checkpoint = CHECKPOINT_ROOT / "best_checkpoint.pt"
history = RUN_ROOT / "training_history.csv"
checks = {
    "bestCheckpointExists": best_checkpoint.is_file(),
    "historyExists": history.is_file(),
    "studyLeakage": bool(set(train_samples.study_id) & set(validation_samples.study_id)),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}
failures = [k for k,v in checks.items() if k not in {"studyLeakage","internalTestAccessed","officialTestAccessed"} and v is not True]
failures += [k for k in ("studyLeakage","internalTestAccessed","officialTestAccessed") if checks[k] is not False]
print(json.dumps(checks, indent=2))
if failures:
    raise RuntimeError("Notebook 56 no habilitado: " + ", ".join(failures))
print({
    "status": "APPROVED_FOR_NOTEBOOK_56",
    "bestCheckpoint": str(best_checkpoint),
    "plannedFinalArtifact": "rsna_central_stenosis_sagittal_t2_2p5d.pt",
})
